# Parakeet ASR on a free Colab GPU → OpenShorts

Hosts **NVIDIA `parakeet-tdt-0.6b-v3`** (via the prebuilt **NeMo-Speech.cpp v0.3.0** CUDA runtime) behind a small FastAPI server and exposes it through a public **Cloudflare quick tunnel**.

**How to use:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Copy the `https://...trycloudflare.com` URL printed by the tunnel cell into OpenShorts.

| Step | Cell |
|---|---|
| 1 | GPU check (fails loudly without a GPU) |
| 2 | Download + sha256-verify + extract the prebuilt runtime (no building) |
| 3 | `nemo-speech pull parakeet-tdt` (714 MB q8_0 GGUF, verified by the CLI) |
| 4 | FastAPI server on `0.0.0.0:8000` |
| 5 | Cloudflare tunnel + public URL |
| 6 | Optional keep-alive monitor |

**API contract**

- `POST /transcribe` — multipart field `file` (16 kHz mono s16le PCM wav). Returns the JSON printed by `nemo-speech transcribe --format json` (`text`, `confidence`, `duration`, `languages`, `words[]`) plus `"backend": "CUDA"`.
- `GET /health` — `{"status": "ok", "backend": "CUDA"}`.

> ⚠️ **Colab realities**
> - Free Colab sessions **disconnect**: an idle timeout, plus a hard cap of roughly **12 hours**. When that happens the server and tunnel die.
> - The **tunnel URL changes on every run** (Cloudflare quick tunnels are ephemeral). Re-paste the new URL into OpenShorts each time.
> - Serving a persistent public endpoint from free Colab **bends Colab's acceptable-use rules**. Treat this notebook as **personal use only**.
> - Every cell is safe to re-run: downloads and extraction are skipped when already done.

In [ ]:
# 1) GPU check — fail loudly if no GPU is attached
import shutil, subprocess

HELP = (
    "No GPU detected.\n"
    "In Colab: Runtime > Change runtime type > Hardware accelerator: T4 GPU, then Save and re-run."
)

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(HELP)

r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(HELP + "\n\nnvidia-smi said:\n" + (r.stderr or r.stdout))

gpu_name, gpu_mem, gpu_driver = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
print(f"GPU    : {gpu_name}")
print(f"VRAM   : {gpu_mem}")
print(f"Driver : {gpu_driver}")

cc = subprocess.run(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if cc.returncode == 0 and cc.stdout.strip():
    print(f"Compute capability: {cc.stdout.strip().splitlines()[0].strip()} (T4 = 7.5, supported)")
print("\nGPU OK")

In [ ]:
# 2) Prebuilt NeMo-Speech.cpp runtime — download, verify sha256, extract (no building, no CUDA toolkit)
import hashlib, os, pathlib, shutil, subprocess

VERSION = "0.3.0"
ARCHIVE = f"nemo-speech-{VERSION}-linux-x86_64-cuda.tar.gz"
BASE_URL = f"https://github.com/NVIDIA/NeMo-Speech.cpp/releases/download/v{VERSION}"
INSTALL_DIR = pathlib.Path("/opt/nemo-speech")
PARTIAL_DIR = pathlib.Path("/opt/nemo-speech.partial")
DL_DIR = pathlib.Path("/content/nemo-dl")
DL_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, check=True):
    """Run a command, streaming combined stdout/stderr into the cell output."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p.returncode


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ffmpeg is not needed (OpenShorts sends 16 kHz mono wav) but is harmless
if shutil.which("ffmpeg") is None:
    run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq ffmpeg"], check=False)

archive_path = DL_DIR / ARCHIVE
sha_path = DL_DIR / (ARCHIVE + ".sha256")
bin_path = INSTALL_DIR / "bin" / "nemo-speech"

if not bin_path.exists():
    # Always fetch the checksum fresh; only re-download the archive when missing or corrupt.
    run(["curl", "-fL", "--retry", "3", "-o", str(sha_path), f"{BASE_URL}/{ARCHIVE}.sha256"])
    expected = sha_path.read_text().split()[0].strip().lower()

    if archive_path.exists() and sha256_of(archive_path) != expected:
        print("Existing archive failed checksum; re-downloading.")
        archive_path.unlink()
    if not archive_path.exists():
        run(["curl", "-fL", "--retry", "3", "-o", str(archive_path), f"{BASE_URL}/{ARCHIVE}"])

    actual = sha256_of(archive_path)
    if actual != expected:
        archive_path.unlink()
        raise RuntimeError(f"sha256 MISMATCH\n expected: {expected}\n actual:   {actual}")
    print(f"sha256 verified: {actual}")

    # Work out whether the tarball has a single top-level directory to strip.
    listing = subprocess.run(["tar", "-tzf", str(archive_path)], capture_output=True, text=True, check=True).stdout.splitlines()
    names = [n[2:] if n.startswith("./") else n for n in listing]
    names = [n for n in names if n and n != "."]
    tops = {n.split("/")[0] for n in names}
    strip = 1 if (len(tops) == 1 and any(n.startswith(next(iter(tops)) + "/") for n in names)) else 0

    if PARTIAL_DIR.exists():
        shutil.rmtree(PARTIAL_DIR)
    PARTIAL_DIR.mkdir(parents=True)
    run(["tar", "-xzf", str(archive_path), "-C", str(PARTIAL_DIR), f"--strip-components={strip}"])
    if INSTALL_DIR.exists():
        shutil.rmtree(INSTALL_DIR)
    PARTIAL_DIR.rename(INSTALL_DIR)
    print(f"Extracted to {INSTALL_DIR}")
else:
    print(f"{bin_path} already present; skipping download/extract.")

# Locate the binary (normally /opt/nemo-speech/bin/nemo-speech)
if not bin_path.exists():
    found = [p for p in INSTALL_DIR.rglob("nemo-speech") if p.is_file() and os.access(p, os.X_OK)]
    if not found:
        raise RuntimeError(f"nemo-speech binary not found under {INSTALL_DIR}")
    bin_dir = found[0].parent
else:
    bin_dir = INSTALL_DIR / "bin"

# PATH (+ bundled CUDA userspace libs, if the archive ships a lib dir)
os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}" if str(bin_dir) not in os.environ["PATH"].split(":") else os.environ["PATH"]
lib_dirs = [str(d) for d in (INSTALL_DIR / "lib", INSTALL_DIR / "lib64") if d.exists()]
if lib_dirs:
    cur = os.environ.get("LD_LIBRARY_PATH", "")
    parts = [p for p in cur.split(":") if p]
    os.environ["LD_LIBRARY_PATH"] = ":".join([d for d in lib_dirs if d not in parts] + parts)

print("\nnemo-speech ->", shutil.which("nemo-speech"))
run(["nemo-speech", "--help"], check=False)

In [ ]:
# 3) Pull the model (714 MB q8_0 GGUF, sha256-verified by the CLI, idempotent)
run(["nemo-speech", "pull", "parakeet-tdt"])
print("\nModel ready: parakeet-tdt")

In [ ]:
# 4) FastAPI server: POST /transcribe, GET /health on 0.0.0.0:8000
run(["pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "requests"])

import json, os, re, shutil, subprocess, tempfile, threading, time, wave
import requests
import uvicorn
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse

MODEL = "parakeet-tdt"
PORT = 8000
BACKEND_RE = re.compile(r"backend=([A-Za-z0-9_+\-]+)")
_gpu_lock = threading.Lock()  # one transcription at a time on the shared T4
state = {"backend": "unknown"}


def run_transcribe(wav_path):
    """Run the CLI; return (payload_dict, backend). Raises RuntimeError on failure."""
    with _gpu_lock:
        p = subprocess.run(
            ["nemo-speech", "transcribe", wav_path, "--model", MODEL, "--format", "json"],
            capture_output=True, text=True, timeout=3600,
        )
    if p.returncode != 0:
        raise RuntimeError(f"nemo-speech exited {p.returncode}: {p.stderr.strip()[-2000:]}")
    m = BACKEND_RE.search(p.stderr)
    backend = m.group(1) if m else "unknown"
    try:
        payload = json.loads(p.stdout)
    except ValueError:
        raise RuntimeError(f"nemo-speech did not return JSON: {p.stdout[:500]!r} / stderr: {p.stderr[-500:]!r}")
    if not isinstance(payload, dict):
        raise RuntimeError("unexpected JSON shape from nemo-speech")
    payload["backend"] = backend  # everything else is exactly what the CLI printed
    return payload, backend


app = FastAPI(title="Parakeet ASR (NeMo-Speech.cpp)")


@app.get("/health")
def health():
    return {"status": "ok", "backend": state["backend"]}


@app.post("/transcribe")
def transcribe(file: UploadFile = File(...)):
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        with open(tmp, "wb") as out:
            shutil.copyfileobj(file.file, out)
        if os.path.getsize(tmp) == 0:
            raise HTTPException(status_code=400, detail="empty upload")
        payload, backend = run_transcribe(tmp)
        state["backend"] = backend
        return JSONResponse(payload)
    except HTTPException:
        raise
    except subprocess.TimeoutExpired:
        raise HTTPException(status_code=504, detail="transcription timed out")
    except RuntimeError as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass


def make_test_wav(path, seconds=1, rate=16000):
    with wave.open(path, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(b"\x00\x00" * rate * seconds)


# Probe once before serving so /health reports the real backend and we fail loudly if the CLI is broken.
probe = "/tmp/probe.wav"
make_test_wav(probe)
_, state["backend"] = run_transcribe(probe)
print("Probe backend:", state["backend"])
if state["backend"].upper() != "CUDA":
    print("\n*** WARNING: backend is not CUDA — transcription is NOT running on the GPU. ***\n")

# Stop a previous server from an earlier run of this cell, then start fresh.
if "_uv_server" in globals() and _uv_thread.is_alive():
    _uv_server.should_exit = True
    _uv_thread.join(timeout=15)

_uv_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info"))
_uv_thread = threading.Thread(target=_uv_server.run, daemon=True)
_uv_thread.start()

for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(0.5)
else:
    raise RuntimeError("server did not come up on port 8000")

print("Health:", requests.get(f"http://127.0.0.1:{PORT}/health").json())
with open(probe, "rb") as f:
    r = requests.post(f"http://127.0.0.1:{PORT}/transcribe", files={"file": ("probe.wav", f, "audio/wav")})
print("Self-test /transcribe ->", r.status_code, r.text[:300])

In [ ]:
# 5) Cloudflare quick tunnel -> public URL for OpenShorts
import os, pathlib, re, subprocess, threading, time
import requests
from IPython.display import HTML, display

CF_BIN = pathlib.Path("/usr/local/bin/cloudflared")
if not CF_BIN.exists():
    run(["curl", "-fL", "--retry", "3", "-o", str(CF_BIN),
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    CF_BIN.chmod(0o755)

# Stop any tunnel from a previous run of this cell
if "_cf_proc" in globals() and _cf_proc.poll() is None:
    _cf_proc.terminate()
    try:
        _cf_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        _cf_proc.kill()
subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)

URL_RE = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
_cf_proc = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
_cf_log, _cf_url = [], []


def _drain():  # keep reading so the pipe never fills up
    for line in _cf_proc.stdout:
        _cf_log.append(line)
        m = URL_RE.search(line)
        if m and not _cf_url:
            _cf_url.append(m.group(0))


threading.Thread(target=_drain, daemon=True).start()

deadline = time.time() + 60
while not _cf_url and time.time() < deadline and _cf_proc.poll() is None:
    time.sleep(0.5)
if not _cf_url:
    raise RuntimeError("No trycloudflare.com URL found. cloudflared output:\n" + "".join(_cf_log[-30:]))

TUNNEL_URL = _cf_url[0]

# Confirm the tunnel actually forwards to the server (DNS can take a few seconds to propagate)
ok = False
for _ in range(30):
    try:
        if requests.get(TUNNEL_URL + "/health", timeout=5).ok:
            ok = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

display(HTML(f"""
<div style="padding:24px;margin:12px 0;border:3px solid #6200EE;border-radius:14px;background:#f5efff;color:#111;font-family:system-ui,sans-serif">
  <div style="font-size:14px;letter-spacing:.08em;text-transform:uppercase;color:#6200EE;font-weight:700">Paste this URL into OpenShorts</div>
  <div style="font-size:30px;font-weight:800;margin:10px 0;word-break:break-all">{TUNNEL_URL}</div>
  <div style="font-size:16px">Transcription endpoint: <code>{TUNNEL_URL}/transcribe</code> &nbsp;|&nbsp; Health: <code>{TUNNEL_URL}/health</code></div>
  <div style="font-size:14px;margin-top:8px;color:#444">{'Tunnel health check passed.' if ok else 'Tunnel is up but the health check has not passed yet - wait a few seconds and open the health URL.'}</div>
</div>
"""))

print("=" * 72)
print("OPENSHORTS TRANSCRIPTION URL:", TUNNEL_URL)
print("=" * 72)
print(f'\nTest from anywhere:\n  curl -F "file=@audio_16k_mono.wav" {TUNNEL_URL}/transcribe')

In [ ]:
# 6) Optional: keep this cell running to monitor the server (stop with the interrupt button).
#    A running cell keeps the session "active", but Colab can still disconnect you at any time.
import time, requests

print("Monitoring. Public URL:", TUNNEL_URL)
last_print = 0
try:
    while True:
        try:
            h = requests.get("http://127.0.0.1:8000/health", timeout=5).json()
            alive = _cf_proc.poll() is None
            if not alive:
                print("cloudflared exited - re-run the tunnel cell for a new URL.")
                break
            if time.time() - last_print > 600:
                print(time.strftime("%H:%M:%S"), "healthy:", h, "|", TUNNEL_URL)
                last_print = time.time()
        except Exception as e:
            print(time.strftime("%H:%M:%S"), "health check failed:", e)
        time.sleep(60)
except KeyboardInterrupt:
    print("Monitor stopped (server and tunnel keep running).")